In [1]:
!pip install -U langchain langchain_openai langchain_community requests python-dotenv beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.1/112.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.0/503.0 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: beautifulsoup4
    Found existing installation: beautifulsoup4 4.13.5
    Uninstalling beautifulsoup4-4

In [12]:
import os

os.environ["OPENROUTER_API_KEY"] = "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"

In [13]:
import os

print(os.getenv("OPENROUTER_API_KEY")[:12])

sk-or-v1-7a7


In [18]:
os.environ["TAVILY_API_KEY"] = "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"

In [19]:
import os
import requests
from bs4 import BeautifulSoup

from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

from langchain_community.tools.tavily_search import TavilySearchResults

In [20]:
internet_search = TavilySearchResults()

/tmp/ipykernel_600/3012731111.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  internet_search = TavilySearchResults()


In [21]:
!pip install -U langchain-tavily

In [24]:
from langchain_tavily import TavilySearch

internet_search = TavilySearch(max_results=3)

In [25]:
internet_search.invoke({"query": "AI in education"})

{'query': 'AI in education',
 'response_time': 1.15,
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.unesco.org/en/digital-education/artificial-intelligence',
   'title': 'Artificial intelligence in education - AI | UNESCO',
   'content': '# Artificial intelligence in education. Artificial Intelligence (AI) has the potential to address some of the biggest challenges in education today, innovate teaching and learning practices, and accelerate progress towards SDG 4. UNESCO is committed to supporting Member States to harness the potential of AI technologies for achieving the Education 2030 Agenda, while ensuring that its application in educational contexts is guided by the core principles of inclusion and equity. Within the framework of the\xa0Beijing Consensus, UNESCO developed\xa0Artificial intelligence and education: Guidance for policy-makers to foster the readiness of education policy-makers in artificial intelligence. Generative AI an

In [39]:
@tool
def fetch_url(url: str) -> str:
    """Fetch readable text content from a URL."""

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)

        if response.status_code != 200:
            return f"FETCH_FAILED: {url}"

        soup = BeautifulSoup(response.text, "html.parser")

        for tag in soup(["script", "style", "noscript"]):
            tag.decompose()

        text = soup.get_text(separator=" ", strip=True)

        if not text or len(text.strip()) < 200:
            return f"FETCH_FAILED: {url}"

        return text[:12000]

    except Exception:
        return f"FETCH_FAILED: {url}"

In [29]:
test_url = "https://en.wikipedia.org/wiki/Artificial_intelligence_in_education"

print(fetch_url.invoke({"url": test_url})[:1000])

Artificial intelligence in education - Wikipedia Jump to content Main menu Main menu move to sidebar hide Navigation Main page Contents Current events Random article About Wikipedia Contact us Contribute Help Learn to edit Community portal Recent changes Upload file Special pages Search Search Appearance Donate Create account Log in Personal tools Donate Create account Log in Contents move to sidebar hide (Top) 1 History 2 Theory Toggle Theory subsection 2.1 Three paradigms of AIEd 2.2 Socio-technical imaginaries 2.3 Post-humanist and new materialist perspectives 3 Applications Toggle Applications subsection 3.1 AI-based tutoring systems 3.2 Generative AI 3.3 Emotional AI 4 Perspectives Toggle Perspectives subsection 4.1 Commercial perspectives 4.2 Institutional perspectives 4.3 Student perspectives 5 Problems Toggle Problems subsection 5.1 Over-reliance, inaccuracy, and academic integrity 5.2 Accessibility 5.3 Bias 5.4 Data privacy and intellectual property 5.5 Invisible labour and en

In [30]:
model = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

In [31]:
response = model.invoke("Say hello in one short sentence.")
print(response.content)

Hello!


In [40]:
system_prompt = """
You are a research agent.

You must follow this exact process:
1. Read the user's question carefully.
2. Use internet_search to find relevant information.
3. Select exactly the top 3 URLs from the search results.
4. Call fetch_url for each of those 3 URLs at most once.
5. Use only successfully fetched page contents in the final answer.
6. If one URL fails, skip it and continue with the remaining successful sources.
7. Do not retry the same failed URL repeatedly.
8. Do not cite any URL that failed to fetch successfully.
9. Write the final answer based on fetched page contents, not just search snippets.
10. Include only the URLs that were fetched successfully in the final answer.
"""

In [41]:
agent = create_agent(
    model=model,
    tools=[internet_search, fetch_url],
    system_prompt=system_prompt,
)

In [43]:
question = "What are the health benefits of coffee?"

response = agent.invoke({
    "messages": [
        {"role": "user", "content": question}
    ]
})

print(response["messages"][-1].content)

**Health Benefits of Coffee – Summary of Evidence**

Coffee is more than a morning stimulant; a growing body of research shows that moderate coffee consumption can have several measurable health benefits. The following points are drawn directly from the content of two successfully fetched sources:

1. **Overall Health & Longevity**  
   - A large analysis of nearly 220 studies (BMJ 2017) found coffee drinkers are **17 % less likely to die early** from any cause, **19 % less likely to die of heart disease**, and **18 % less likely to develop cancer** compared with non‑drinkers.  
   - Moderate coffee intake (about 1.5–3.5 cups per day) is associated with a **30 % lower risk of death** from any cause (Annals of Internal Medicine 2022).  

2. **Type 2 Diabetes**  
   - Increasing coffee consumption by just one cup per day can lower the risk of developing type 2 diabetes by **11 %**; decreasing intake raises the risk by **17 %** (Harvard study, cited by Rush).  
   - However, people who al

In [44]:
print(response["messages"][-1].content)

**Health Benefits of Coffee – Summary of Evidence**

Coffee is more than a morning stimulant; a growing body of research shows that moderate coffee consumption can have several measurable health benefits. The following points are drawn directly from the content of two successfully fetched sources:

1. **Overall Health & Longevity**  
   - A large analysis of nearly 220 studies (BMJ 2017) found coffee drinkers are **17 % less likely to die early** from any cause, **19 % less likely to die of heart disease**, and **18 % less likely to develop cancer** compared with non‑drinkers.  
   - Moderate coffee intake (about 1.5–3.5 cups per day) is associated with a **30 % lower risk of death** from any cause (Annals of Internal Medicine 2022).  

2. **Type 2 Diabetes**  
   - Increasing coffee consumption by just one cup per day can lower the risk of developing type 2 diabetes by **11 %**; decreasing intake raises the risk by **17 %** (Harvard study, cited by Rush).  
   - However, people who al